In [1]:
import mlx.core as mx
import numpy as np
import time
import matplotlib.pyplot as plt
import mlx
import scipy
print(mx.metal.is_available())

True


In [2]:
RESOLUTION = 2**4
def prodsine(x: mx.array,func: mx.array,fpow: int):
    NUMINPUT = x.shape[0]
    NUMDIM = x.shape[1]
    NUMFREQ = func.shape[2]
    if NUMFREQ*NUMINPUT*NUMDIM>2000000000:
        print(" maybe this needs to be cut into smaller chunks {} {}".format(x.shape,func.shape))
    if x.shape[1]!=func.shape[1]:
        print(" x shape {} must be the same as func shape {} ".format(x.shape,func.shape))
    if len(x.shape)!=3:
        print(" x shape {} length should be 3 ".format(x.shape))
    # maybe I should have a check here to make sure it isn't too large
    mult_out = (x*func/RESOLUTION*2)*np.pi
    mult_out = (mx.cos(mult_out)+1j*mx.sin(mult_out))*fpow
    #print(" this is all the sin {} with shape {}".format(mult_out,mult_out.shape))
    mult_out = mx.prod(mult_out,axis=1)
    #print(" now the different components {} with shape {}".format(mult_out,mult_out.shape))
    mult_out = mx.sum(mult_out,axis=1)
    #print(" nwo is the different the sum of {} ".format(mult_out))
    return mult_out

In [ ]:
NUMDIM=2
SPARFREQ = 2**2
TrueFreq = mx.random.randint(low=0,high=RESOLUTION,shape=[1,NUMDIM,SPARFREQ],dtype=mx.int32)
TruePow = mx.random.randint(low=1,high=SPARFREQ,shape=[SPARFREQ],dtype=mx.int32)
TruePow = mx.arange(1,SPARFREQ+1)
print(TrueFreq)
print(TruePow)

In [ ]:
firstarray = mx.zeros((NUMDIM*2*RESOLUTION,NUMDIM,1))
print(firstarray.shape)
firstarray[:RESOLUTION,0,0]=mx.arange(RESOLUTION)
firstarray[RESOLUTION:2*RESOLUTION,1,0]=mx.arange(RESOLUTION)
firstarray[2*RESOLUTION:3*RESOLUTION,1,0]=mx.arange(RESOLUTION)
firstarray[2*RESOLUTION:3*RESOLUTION,0,0]=mx.arange(RESOLUTION)
firstarray[3*RESOLUTION:,1,0]=mx.arange(RESOLUTION)
firstarray[3*RESOLUTION:,0,0]=RESOLUTION-mx.arange(RESOLUTION)-1
print(firstarray.T)

In [ ]:
output = prodsine(firstarray,TrueFreq,TruePow)/SPARFREQ

In [ ]:
print(output.shape)
print(output)

In [ ]:
fout1 = mx.fft.fft(output[:RESOLUTION]) 
fout2 = mx.fft.fft(output[RESOLUTION:2*RESOLUTION]) 
fout3 = mx.fft.fft(output[2*RESOLUTION:3*RESOLUTION]) 
fout4 = mx.fft.fft(output[3*RESOLUTION:]) 
print(type(fout1))
print(fout1.shape)

In [ ]:
plt.plot(np.real(np.array(fout1)))
plt.plot(np.real(np.array(fout2)))
plt.plot(np.real(np.array(fout3)))
plt.plot(np.real(np.array(fout4)))

In [ ]:
plt.plot(np.real(np.array(fout3)))


In [ ]:
plt.plot(np.real(np.array(fout2)))
plt.plot(np.real(np.array(fout1)))

In [ ]:
plt.plot(np.imag(np.array(fout1)))
plt.plot(np.imag(np.array(fout2)))
plt.plot(np.imag(np.array(fout3)))
plt.plot(np.imag(np.array(fout4)))

In [ ]:
xlarge = mx.zeros((RESOLUTION*RESOLUTION,NUMDIM,1))
for i in range(RESOLUTION):
    for j in range(RESOLUTION):
        xlarge[i+j*RESOLUTION,0,0]=i
        xlarge[i+j*RESOLUTION,1,0]=j
print(xlarge.T)
largeoutput = prodsine(xlarge,TrueFreq,TruePow)/SPARFREQ
print(largeoutput.shape)
print(xlarge[:,0,0].reshape(RESOLUTION,RESOLUTION))
print(xlarge[:,1,0].reshape(RESOLUTION,RESOLUTION))
print(largeoutput.reshape(RESOLUTION,RESOLUTION))
largeft = mx.fft.fft2(largeoutput.reshape(RESOLUTION,RESOLUTION))+.001

In [ ]:
plt.imshow(np.log(np.real(np.array(largeft))))
plt.colorbar()

In [ ]:
plt.imshow(scipy.ndimage.rotate(np.log(np.real(mx.array(largeft))),angle=90))
plt.colorbar()

In [ ]:
plt.imshow(scipy.ndimage.rotate(np.log(np.real(mx.array(largeft))),angle=45))
plt.colorbar()

In [ ]:
plt.imshow(np.abs(np.array(largeoutput.reshape(RESOLUTION,RESOLUTION))))

In [ ]:
A=np.zeros((SPARFREQ*SPARFREQ,2*NUMDIM*SPARFREQ),dtype='float64')
B=np.zeros((2*NUMDIM*SPARFREQ),dtype='float64')
D1dict = {}
D2dict = {}
print(A.shape)
print(B.shape)

In [ ]:
counter=0
for j in range(RESOLUTION):
    #print(j)
    if np.real(np.array(fout1))[j] < 1:
        continue
    print(j)
    B[counter]=np.real(np.array(fout1))[j]
    for i in range(SPARFREQ):
        #print(i)
        A[i+counter*SPARFREQ,counter]=1
    D1dict[j] = counter
    counter += 1
print(A)

In [ ]:
counter2=0
for j in range(RESOLUTION):
    #print(j)
    if np.real(np.array(fout2))[j] < 1:
        continue
    print(j)
    B[SPARFREQ+counter2]=np.real(np.array(fout2))[j]
    for i in range(SPARFREQ):
        #print(i)
        A[counter2+i*SPARFREQ,counter+counter2]=1
    D2dict[j]=counter2
    counter2 += 1
print(A)
print(B)

In [ ]:
counter3=0
print(TrueFreq)
print(D1dict)
print(D2dict)
for j in range(RESOLUTION):
    #print(j)
    if np.real(np.array(fout3))[j] < 1:
        continue
    print(" <> this is the fout3 points {}".format(j))
    B[SPARFREQ+SPARFREQ+counter3]=np.real(np.array(fout3))[j]
    for i in range(int(RESOLUTION/2)):
        #print(i,j)
        cim = j-i        
        cip = j+i
        #if cim<=0:
        #    cim += RESOLUTION
        #if cip>=RESOLUTION:
        #    cip -= RESOLUTION
        if cip==RESOLUTION:
            cip=0
        if cim==RESOLUTION:
            cim=0
        #print(cim,cip)
        print(" looking for this one {} {}".format(cim,cip))
        check1 = (cim in D1dict.keys() and cip in D2dict.keys()) or (cim in D2dict.keys() and cip in D1dict.keys())
        check2 = (cim in D2dict.keys() and cip+1 in D1dict.keys()) or (cim in D2dict.keys() and cip+1 in D1dict.keys())
        check3 = (cim-1 in D2dict.keys() and cip in D1dict.keys()) or (cim-1 in D2dict.keys() and cip in D1dict.keys())
        check4 = (cim+1 in D2dict.keys() and cip in D1dict.keys()) or (cim+1 in D2dict.keys() and cip in D1dict.keys())
        check5 = (cim in D2dict.keys() and cip-1 in D1dict.keys()) or (cim in D2dict.keys() and cip-1 in D1dict.keys())
        if check1:
            print(" found this one {} {}".format(cim,cip))
    counter3 +=1
print(A)
print(B)

In [ ]:
np.linalg.lstsq(A.T, B, rcond='warn')